<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_06_window_scaling_seq2seq/stage_06_window_scaling_seq2seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_06_window_scaling_seq2seq**




## **0. Configuración del Entorno**


### 0.1. Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación de librerías


In [2]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [3]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import joblib
import os
import json
import logging
from pathlib import Path
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd

# ----------------------------
# Logging
# ----------------------------
logging.basicConfig(
    level=os.environ.get("LOG_LEVEL", "INFO"),
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger("stage_06_window_scaling_seq2seq")

### 0.4. Definición de rutas

In [19]:
# ============================================================
# Paths / IO (via env o defaults)
# ============================================================
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

IN_PARQUET_TRAIN = Path(os.environ.get("IN_PARQUET_TRAIN", "data/splits/mnq_train.parquet"))
IN_PARQUET_VALID = Path(os.environ.get("IN_PARQUET_VALID", "data/splits/mnq_valid.parquet"))
IN_PARQUET_TEST = Path(os.environ.get("IN_PARQUET_TEST", "data/splits/mnq_test.parquet"))

IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/stage_04_feature_engineering_summary.json"))
IN_ARTIFACT_GESTATION = Path(os.environ.get("IN_ARTIFACT_GESTATION", "reports/stage_03b_target_definition_summary.json"))



OUT_WINDOWS_60_TRAIN = Path(os.environ.get("OUT_WINDOWS_60", "data/windows/windows_train_60.npz"))
OUT_WINDOWS_60_VALID = Path(os.environ.get("OUT_WINDOWS_60", "data/windows/windows_valid_60.npz"))
OUT_WINDOWS_60_TEST  = Path(os.environ.get("OUT_WINDOWS_60", "data/windows/windows_test_60.npz"))

OUT_WINDOWS_90_TRAIN = Path(os.environ.get("OUT_WINDOWS_90", "data/windows/windows_train_90.npz"))
OUT_WINDOWS_90_VALID = Path(os.environ.get("OUT_WINDOWS_90", "data/windows/windows_valid_90.npz"))
OUT_WINDOWS_90_TEST  = Path(os.environ.get("OUT_WINDOWS_90", "data/windows/windows_test_90.npz"))

# Escalados

OUT_WINDOWS_60_TRAIN_Z = Path(os.environ.get("OUT_WINDOWS_60_Z", "data/windows/scaled/windows_train_60_z.npz"))
OUT_WINDOWS_60_VALID_Z = Path(os.environ.get("OUT_WINDOWS_60_Z", "data/windows/scaled/windows_valid_60_z.npz"))
OUT_WINDOWS_60_TEST_Z  = Path(os.environ.get("OUT_WINDOWS_60_Z", "data/windows/scaled/windows_test_60_z.npz"))

OUT_WINDOWS_90_TRAIN_Z = Path(os.environ.get("OUT_WINDOWS_90_Z", "data/windows/scaled/windows_train_90_z.npz"))
OUT_WINDOWS_90_VALID_Z = Path(os.environ.get("OUT_WINDOWS_90_Z", "data/windows/scaled/windows_valid_90_z.npz"))
OUT_WINDOWS_90_TEST_Z  = Path(os.environ.get("OUT_WINDOWS_90_Z", "data/windows/scaled/windows_test_90_z.npz"))

OUT_SCALER_60 = Path(os.environ.get("OUT_SCALER", "data/windows/scaled/scaler_60.pkl"))
OUT_SCALER_90 = Path(os.environ.get("OUT_SCALER", "data/windows/scaled/scaler_90.pkl"))

OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/windows_scaled_summary.json"))


#PARA NOTEBOOK

IN_PARQUET_TRAIN = DRIVE_DIR / IN_PARQUET_TRAIN
IN_PARQUET_VALID = DRIVE_DIR / IN_PARQUET_VALID
IN_PARQUET_TEST = DRIVE_DIR / IN_PARQUET_TEST
IN_ARTIFACT = DRIVE_DIR / IN_ARTIFACT
IN_ARTIFACT_GESTATION = DRIVE_DIR / IN_ARTIFACT_GESTATION
OUT_WINDOWS_60_TRAIN = DRIVE_DIR / OUT_WINDOWS_60_TRAIN
OUT_WINDOWS_60_VALID = DRIVE_DIR / OUT_WINDOWS_60_VALID
OUT_WINDOWS_60_TEST = DRIVE_DIR / OUT_WINDOWS_60_TEST

OUT_WINDOWS_90_TRAIN = DRIVE_DIR / OUT_WINDOWS_90_TRAIN
OUT_WINDOWS_90_VALID = DRIVE_DIR / OUT_WINDOWS_90_VALID
OUT_WINDOWS_90_TEST = DRIVE_DIR / OUT_WINDOWS_90_TEST

OUT_WINDOWS_60_TRAIN_Z = DRIVE_DIR / OUT_WINDOWS_60_TRAIN_Z
OUT_WINDOWS_60_VALID_Z = DRIVE_DIR / OUT_WINDOWS_60_VALID_Z
OUT_WINDOWS_60_TEST_Z = DRIVE_DIR / OUT_WINDOWS_60_TEST_Z

OUT_WINDOWS_90_TRAIN_Z = DRIVE_DIR / OUT_WINDOWS_90_TRAIN_Z
OUT_WINDOWS_90_VALID_Z = DRIVE_DIR / OUT_WINDOWS_90_VALID_Z
OUT_WINDOWS_90_TEST_Z = DRIVE_DIR / OUT_WINDOWS_90_TEST_Z

OUT_SCALER_60 = DRIVE_DIR / OUT_SCALER_60
OUT_SCALER_90 = DRIVE_DIR / OUT_SCALER_90

OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY



## **1. Carga de datos**

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [13]:
def load_mnq_parquet(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el parquet de entrada: {path}")

    log.info("[OK] Cargando parquet: %s", path)

    df = pd.read_parquet(path)

    # Asegurar DatetimeIndex
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    # Orden temporal explícito
    df = df.sort_index()

    return df


In [14]:
mnq_train = load_mnq_parquet(IN_PARQUET_TRAIN)
mnq_valid = load_mnq_parquet(IN_PARQUET_VALID)
mnq_test = load_mnq_parquet(IN_PARQUET_TEST)

### 1.2. Información de datasets


In [15]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [16]:
info_dataset(mnq_train, 'mnq_train')
info_dataset(mnq_valid, 'mnq_valid')
info_dataset(mnq_test, 'mnq_test')

Información del dataset mnq_train:

	Cantidad de días: 912
	Registros por día: 421
	Hora diaria de inicio 07:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York

Información del dataset mnq_valid:

	Cantidad de días: 195
	Registros por día: 421
	Hora diaria de inicio 07:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York

Información del dataset mnq_test:

	Cantidad de días: 196
	Registros por día: 421
	Hora diaria de inicio 07:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York



(196, np.float64(421.0))

### 1.3. Carga de listado de features y targets

In [18]:
import json
with open(IN_ARTIFACT, "r", encoding="utf-8") as f:
    features_target_summary = json.load(f)

features = features_target_summary["details"]["schema"]["features"]
targets = features_target_summary["details"]["schema"]["targets"]

print("Features:", features)
print("Targets:", targets)

Features: ['open', 'high', 'low', 'close', 'volume', 'price_ema60', 'momentum_10', 'roc_30', 'roc_60']
Targets: ['delta_pts_60', 'delta_pts_90']


In [21]:
import json
with open(IN_ARTIFACT_GESTATION, "r", encoding="utf-8") as f:
    target_definition_summary = json.load(f)



In [23]:
gestation_window_start = target_definition_summary["details"]["gestation_window"]["start_hhmm"]
gestation_window_end = target_definition_summary["details"]["gestation_window"]["end_hhmm"]


In [25]:
gestation_window_end

'08:49'

## **2. Carga de features para cada horizonte 60 y 90min**

Definimos el target de cada horizonte:

In [ ]:
target_60 = targets[0]
target_90 = targets[1]

Luego definimos el listado de features para cada horizonte:

In [27]:
features

['open',
 'high',
 'low',
 'close',
 'volume',
 'price_ema60',
 'momentum_10',
 'roc_30',
 'roc_60']

In [28]:
features_60 = [f for f in features if f != "roc_30"]
features_90 = [f for f in features if f != "roc_60"]

In [29]:
features_60

['open',
 'high',
 'low',
 'close',
 'volume',
 'price_ema60',
 'momentum_10',
 'roc_60']

In [30]:
features_90

['open',
 'high',
 'low',
 'close',
 'volume',
 'price_ema60',
 'momentum_10',
 'roc_30']

## **3. Filtrado de datasets ajustado a la ventana de gestación**

Anteriormente habíamos mencionado que entrenariamos el modelo con los datos de la ventana de gestación.

In [31]:
gestation_window_start, gestation_window_end

('08:21', '08:49')

En este punto, es necesario filtrar los dataset dentro de esos horarios:


In [32]:
def filter_gestation_window(
    df: pd.DataFrame,
    start: str = gestation_window_start,
    end: str = gestation_window_end,
    date_col: str = "date",
) -> pd.DataFrame:
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El DataFrame debe tener un DatetimeIndex en df.index.")

    if date_col not in df.columns:
        raise KeyError(f"No se encontró la columna '{date_col}' en el DataFrame.")

    # 1) Orden temporal antes de filtrar
    df = df.sort_index()

    # 2) Consistencia: date_col debe coincidir con la fecha del índice
    # (esto previene 'mezclar días' por datos corruptos/desalineados)
    idx_dates = df.index.date
    col_dates = pd.to_datetime(df[date_col]).dt.date
    if not (col_dates.values == idx_dates).all():
        raise ValueError(
            f"Inconsistencia entre '{date_col}' y la fecha del DatetimeIndex. "
            "Revise que no haya registros mal asignados a otro día."
        )

    # 3) Filtrado por ventana horaria (no cruza días; opera dentro de cada fecha)
    out = df.between_time(start, end, inclusive="both")

    # 4) Orden temporal después de filtrar
    out = out.sort_index()

    # 5) Validación adicional: por cada 'date', el índice debe pertenecer a ese mismo día
    out_idx_dates = out.index.date
    out_col_dates = pd.to_datetime(out[date_col]).dt.date
    if not (out_col_dates.values == out_idx_dates).all():
        raise ValueError(
            f"El filtrado generó inconsistencia entre '{date_col}' y el índice. "
            "Esto no debería ocurrir; revise el dataset."
        )

    return out


In [33]:
mnq_train_g = filter_gestation_window(mnq_train)
mnq_valid_g = filter_gestation_window(mnq_valid)
mnq_test_g  = filter_gestation_window(mnq_test)

In [34]:
n_days_train, median_by_day_train = info_dataset(mnq_train_g, 'mnq_train_g')
n_days_valid, median_by_day_valid = info_dataset(mnq_valid_g, 'mnq_valid_g')
n_days_test, median_by_day_test = info_dataset(mnq_test_g, 'mnq_test_g')

Información del dataset mnq_train_g:

	Cantidad de días: 912
	Registros por día: 29
	Hora diaria de inicio 08:21
	Hora diaria de final 08:49
	Zona horaria: America/New_York

Información del dataset mnq_valid_g:

	Cantidad de días: 195
	Registros por día: 29
	Hora diaria de inicio 08:21
	Hora diaria de final 08:49
	Zona horaria: America/New_York

Información del dataset mnq_test_g:

	Cantidad de días: 196
	Registros por día: 29
	Hora diaria de inicio 08:21
	Hora diaria de final 08:49
	Zona horaria: America/New_York



In [35]:
n_days_train, median_by_day_train

(912, np.float64(29.0))

## **4. Definición de windows size**

El tamaño de la ventana de entrada se fija en la cantidad de  registros por día, porque corresponde a la ventana de gestación del movimiento, es decir, el intervalo intradía en el que el mercado concentra la información relevante previa a la expansión del precio.

Al filtrar previamente los datos por esta franja horaria y construir una única secuencia por día, cada muestra representa un contexto temporal homogéneo, evita mezclar dinámicas de distintos momentos de la sesión y alinea la longitud de la secuencia con la lógica operativa del problema, no con el horizonte del target ni con el período de cálculo de los indicadores técnicos.


In [36]:
if median_by_day_train == median_by_day_valid == median_by_day_test:
    window_size = int(median_by_day_train)
    print(f"window_size: {window_size}")
else:
    raise ValueError(
        f"Window size mismatch: "
        f"train={median_by_day_train}, "
        f"valid={median_by_day_valid}, "
        f"test={median_by_day_test}"
    )

window_size: 29


## **5. Generación de ventanas**

Los datasets `mnq_train_g`, `mnq_valid_g` y `mnq_test_g` se encuentran preagrupados por día, de modo que cada día corresponde exactamente a una única ventana temporal fija, sin generación de ventanas deslizantes dentro de la jornada.

- Agrupamiento diario:

  Cada muestra del dataset corresponde a un día de operación.
  Cada día contiene exactamente 30 registros consecutivos minuto a minuto, comprendidos entre las 08:20 y las 08:49 (America/New_York).

- Estructura de las features:

  Para cada día, los 30 registros forman una matriz de features de dimensión 30x7, donde cada fila representa un minuto y cada columna una variable de entrada.

- Definición del target:

  El target ya está definido a nivel de cada fila del dataset y representa el delta de puntos hacia adelante (por ejemplo, a 60 o 90 minutos).

  Para cada día, el objetivo del modelo es predecir el bloque completo de 30 valores de target, alineados uno a uno con los 30 registros de entrada.

  De este modo, cada muestra se define como:

  Entrada (30x8) ⟶ Salida (1x30)

  sin superposición entre días, sin ventanas internas deslizantes y preservando estrictamente la coherencia temporal.


### **5.1. Funciones para construcción de ventanas secuencia a secuencia**

#### **5.1.1. Construcción de ventanas**

In [37]:
import numpy as np

def build_daily_seq2seq_windows(
    df,
    features,
    target_col,
    window_size: int,
    date_col: str = "date",
):
    """
    Construye muestras alineadas con un esquema SEQ2SEQ diario:

        Entrada  X: (window_size, n_features)  -> 30 x 8
        Salida   y: (window_size,)             -> 1 x 30

    Supuestos clave (alineados con el pipeline actual):
    - El dataset ya viene preagrupado por día.
    - Cada día contiene exactamente `window_size` registros consecutivos
      (por ejemplo, 30 minutos entre 08:20 y 08:49).
    - NO se generan ventanas deslizantes dentro del día.
    - Se obtiene UNA muestra por día.

    Parámetros:
    - df: DataFrame que contiene columnas [date_col] + features + target_col
    - features: lista de columnas de entrada
      (por ejemplo: features_60 o features_90)
    - target_col: columna objetivo
      (por ejemplo: delta_pts_60 o delta_pts_90)
    - window_size: cantidad de registros por día (default: 30)
    - date_col: columna usada para agrupar por día (default: "date")

    Retorna:
    - X: np.ndarray con shape (n_dias_validos, window_size, n_features)
    - y: np.ndarray con shape (n_dias_validos, window_size)
    """
    X, y = [], []

    # 1) Agrupamiento diario: cada grupo representa una ventana fija del día
    for _, grupo in df.groupby(date_col):
        # Asegurar orden temporal dentro del día
        grupo = grupo.sort_index()

        # 2) Validar que el día tenga exactamente window_size registros
        if len(grupo) != window_size:
            continue

        # 3) Extraer la matriz de features (30 x n_features)
        X_day = grupo[features].values

        # 4) Extraer el bloque completo de targets (30,)
        y_day = grupo[target_col].values

        # 5) Chequeo de NaN para evitar muestras inválidas
        if np.isnan(X_day).any() or np.isnan(y_day).any():
            continue

        X.append(X_day)
        y.append(y_day)

    return np.array(X), np.array(y)

#### **5.1.2. Construcción de ventanas para train, valid y test**

In [38]:
import os
import numpy as np
from pathlib import Path


def prepare_or_load_seq2seq_windows(
    mnq_train,
    mnq_valid,
    mnq_test,
    features,
    target_col: str,
    window_size: int,
    out_windows_train,
    out_windows_valid,
    out_windows_test,
    date_col: str = "date",
):
    """
    Genera (si no existe) o carga (si ya existe) las ventanas SEQ2SEQ diarias
    usando `build_daily_seq2seq_windows`, y las guarda como .npz comprimido.

    Cada muestra:
      X: (window_size, n_features)
      y: (window_size,)

    Nota: out_windows_* puede ser str o pathlib.Path (por ejemplo PosixPath).
    """

    def _to_path(p) -> Path:
        return p if isinstance(p, Path) else Path(str(p))

    def _ensure_parent_dir(path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)

    def _load_or_build(split_name: str, df, out_path):
        out_path = _to_path(out_path)
        _ensure_parent_dir(out_path)

        if not out_path.exists():
            print(f"No existe -> Generando {split_name} para '{target_col}' y guardando en: {out_path}")
            X, y = build_daily_seq2seq_windows(
                df=df,
                features=features,
                target_col=target_col,
                window_size=window_size,
                date_col=date_col,
            )
            np.savez_compressed(out_path, X=X, y=y)
            print(f"Guardado: {out_path} | X: {X.shape} | y: {y.shape}")
            return X, y

        print(f"Ya existe -> Cargando {split_name} desde: {out_path}")
        data = np.load(out_path)
        X, y = data["X"], data["y"]
        print(f"Cargado: {out_path} | X: {X.shape} | y: {y.shape}")
        return X, y

    X_train, y_train = _load_or_build("train", mnq_train, out_windows_train)
    X_valid, y_valid = _load_or_build("valid", mnq_valid, out_windows_valid)
    X_test,  y_test  = _load_or_build("test",  mnq_test,  out_windows_test)

    return X_train, y_train, X_valid, y_valid, X_test, y_test


#### **5.1.3. Función para revisar composición de ventanas**

In [39]:
import numpy as np

def xy_info_seq2seq(
    horizon_min: int,
    X_train, y_train,
    X_valid, y_valid,
    X_test,  y_test,
    n_features: int | None = None,
):
    """
    Imprime información resumida de los arrays X/y para el esquema diario SEQ2SEQ.

    Esperado (nuestro proyecto):
      X: (n_samples, window_size, n_features)   -> por ejemplo (912, 30, 8)
      y: (n_samples, window_size)              -> por ejemplo (912, 30)

    Parámetros:
    - horizon_min: horizonte del target (60 o 90), solo para rotular
    - n_features: opcional; si se pasa, valida que X.shape[2] coincida
    """

    print(f"Información X/y para horizonte {horizon_min} min:")

    for name, X, y in [
        ("entrenamiento", X_train, y_train),
        ("validación",    X_valid, y_valid),
        ("testeo",        X_test,  y_test),
    ]:
        print(f"\nSet de {name}:")

        # --- Formas ---
        print(f"\tX shape: {X.shape}")
        print(f"\ty shape: {y.shape}")

        # --- Validaciones básicas ---
        if X.ndim != 3:
            print("\t[AVISO] X no es 3D. Se esperaba (n_samples, window_size, n_features).")
        else:
            n_samples, window_size, n_feat = X.shape
            print(f"\t{n_samples} días/muestras (n_samples).")
            print(f"\t{window_size} pasos temporales por día (window_size).")
            print(f"\t{n_feat} features por paso.")

            if n_features is not None and n_feat != n_features:
                print(f"\t[AVISO] n_features esperado={n_features}, encontrado={n_feat}.")

        if y.ndim != 2:
            print("\t[AVISO] y no es 2D. Se esperaba (n_samples, window_size).")
        else:
            if X.shape[0] != y.shape[0]:
                print(f"\t[AVISO] n_samples difiere: X={X.shape[0]} vs y={y.shape[0]}.")
            if X.ndim == 3 and X.shape[1] != y.shape[1]:
                print(f"\t[AVISO] window_size difiere: X={X.shape[1]} vs y={y.shape[1]}.")

        # --- Estadísticas de y (sobre TODOS los valores del bloque) ---
        y_flat = np.asarray(y).ravel()
        print(
            "\tDistribución y (flatten): "
            f"mean={y_flat.mean():.6f}, std={y_flat.std():.6f}, "
            f"min={y_flat.min():.6f}, max={y_flat.max():.6f}"
        )


### **5.2. Generación de ventanas (H = 60min)**

In [40]:
X_train_60, y_train_60, X_valid_60, y_valid_60, X_test_60, y_test_60 = prepare_or_load_seq2seq_windows(
    mnq_train=mnq_train_g,
    mnq_valid=mnq_valid_g,
    mnq_test=mnq_test_g,
    features=features_60,
    target_col="delta_pts_60",
    window_size=window_size,
    out_windows_train=OUT_WINDOWS_60_TRAIN,
    out_windows_valid=OUT_WINDOWS_60_VALID,
    out_windows_test=OUT_WINDOWS_60_TEST,
    date_col="date",
)

Ya existe -> Cargando train desde: /content/drive/MyDrive/neural_profit/data/windows/windows_train_60.npz
Cargado: /content/drive/MyDrive/neural_profit/data/windows/windows_train_60.npz | X: (912, 29, 8) | y: (912, 29)
Ya existe -> Cargando valid desde: /content/drive/MyDrive/neural_profit/data/windows/windows_valid_60.npz
Cargado: /content/drive/MyDrive/neural_profit/data/windows/windows_valid_60.npz | X: (195, 29, 8) | y: (195, 29)
Ya existe -> Cargando test desde: /content/drive/MyDrive/neural_profit/data/windows/windows_test_60.npz
Cargado: /content/drive/MyDrive/neural_profit/data/windows/windows_test_60.npz | X: (196, 29, 8) | y: (196, 29)


In [41]:
xy_info_seq2seq(
    horizon_min=60,
    X_train=X_train_60, y_train=y_train_60,
    X_valid=X_valid_60, y_valid=y_valid_60,
    X_test=X_test_60,   y_test=y_test_60,
    n_features=len(features_60),
)

Información X/y para horizonte 60 min:

Set de entrenamiento:
	X shape: (912, 29, 8)
	y shape: (912, 29)
	912 días/muestras (n_samples).
	29 pasos temporales por día (window_size).
	8 features por paso.
	Distribución y (flatten): mean=-0.085848, std=53.230957, min=-484.000000, max=475.000000

Set de validación:
	X shape: (195, 29, 8)
	y shape: (195, 29)
	195 días/muestras (n_samples).
	29 pasos temporales por día (window_size).
	8 features por paso.
	Distribución y (flatten): mean=0.540539, std=50.131244, min=-284.250000, max=256.500000

Set de testeo:
	X shape: (196, 29, 8)
	y shape: (196, 29)
	196 días/muestras (n_samples).
	29 pasos temporales por día (window_size).
	8 features por paso.
	Distribución y (flatten): mean=-1.192470, std=73.556760, min=-404.000000, max=542.500000


### **5.3. Generación de ventanas (H = 90min)**

In [42]:
X_train_90, y_train_90, X_valid_90, y_valid_90, X_test_90, y_test_90 = prepare_or_load_seq2seq_windows(
    mnq_train=mnq_train_g,
    mnq_valid=mnq_valid_g,
    mnq_test=mnq_test_g,
    features=features_90,
    target_col="delta_pts_90",
    window_size=window_size,
    out_windows_train=OUT_WINDOWS_90_TRAIN,
    out_windows_valid=OUT_WINDOWS_90_VALID,
    out_windows_test=OUT_WINDOWS_90_TEST,
    date_col="date",
)

Ya existe -> Cargando train desde: /content/drive/MyDrive/neural_profit/data/windows/windows_train_90.npz
Cargado: /content/drive/MyDrive/neural_profit/data/windows/windows_train_90.npz | X: (912, 29, 8) | y: (912, 29)
Ya existe -> Cargando valid desde: /content/drive/MyDrive/neural_profit/data/windows/windows_valid_90.npz
Cargado: /content/drive/MyDrive/neural_profit/data/windows/windows_valid_90.npz | X: (195, 29, 8) | y: (195, 29)
Ya existe -> Cargando test desde: /content/drive/MyDrive/neural_profit/data/windows/windows_test_90.npz
Cargado: /content/drive/MyDrive/neural_profit/data/windows/windows_test_90.npz | X: (196, 29, 8) | y: (196, 29)


In [43]:
xy_info_seq2seq(
    horizon_min=90,
    X_train=X_train_90, y_train=y_train_90,
    X_valid=X_valid_90, y_valid=y_valid_90,
    X_test=X_test_90,   y_test=y_test_90,
    n_features=len(features_90),
)

Información X/y para horizonte 90 min:

Set de entrenamiento:
	X shape: (912, 29, 8)
	y shape: (912, 29)
	912 días/muestras (n_samples).
	29 pasos temporales por día (window_size).
	8 features por paso.
	Distribución y (flatten): mean=0.486180, std=78.851648, min=-537.000000, max=570.000000

Set de validación:
	X shape: (195, 29, 8)
	y shape: (195, 29)
	195 días/muestras (n_samples).
	29 pasos temporales por día (window_size).
	8 features por paso.
	Distribución y (flatten): mean=-0.142440, std=70.911722, min=-246.750000, max=295.250000

Set de testeo:
	X shape: (196, 29, 8)
	y shape: (196, 29)
	196 días/muestras (n_samples).
	29 pasos temporales por día (window_size).
	8 features por paso.
	Distribución y (flatten): mean=-1.595048, std=121.641657, min=-422.750000, max=1301.250000


## **6. Escalado de ventanas**

En este punto se escalan las ventanas de entrada para que todas las features tengan la misma magnitud, entrenando el scaler con los datos de entrenamiento y aplicándolo luego a validación y test.

#### **6.1.1. Función para elegir escalador**


In [44]:
def _choose_scaler(scaler_type="standard"):
    st = scaler_type.lower()
    if st in ["standard", "z", "zscore"]:
        return StandardScaler()
    elif st in ["minmax", "min_max"]:
        return MinMaxScaler()
    else:
        raise ValueError("scaler_type debe ser 'standard' o 'minmax'")

#### **6.1.2. Función para entrenar un scaler en datos secuenciales 3D (ventanas), tratándolos como una sola tabla 2D de features.**

In [45]:
# -------------------------------------------------------------------------
# _fit_on_3d
#
# Entrada: X_train_3d con forma (n, W, F)
#   n = número de muestras (ventanas)
#   W = lookback (número de pasos en cada ventana)
#   F = número de features por paso
#
# Qué hace:
# - Aplana las dos primeras dimensiones (n, W) → queda una matriz de (n*W, F).
# - Convierte todas las secuencias en un dataset tabular de features.
# - Ajusta el scaler (ej. StandardScaler) sobre todos los valores de todas
#   las ventanas y pasos, feature por feature.
#
# Resultado: devuelve un scaler entrenado con la estadística global de cada
# feature (media, std, min, max, según el tipo de scaler utilizado).
# -------------------------------------------------------------------------

def _fit_on_3d(X_train_3d, scaler):
    n, W, F = X_train_3d.shape
    scaler.fit(X_train_3d.reshape(-1, F))
    return scaler

#### **6.1.3. Función para aplicar el scaler de _fit_on_3d y devolver los datos escalados, manteniendo la estructura original (n, W, F).**

In [46]:
# -------------------------------------------------------------------------
# _transform_3d
#
# Entrada: X_3d con forma (n, W, F)
#   n = número de muestras (ventanas)
#   W = lookback (número de pasos en cada ventana)
#   F = número de features por paso
#
# Qué hace:
# - Aplana las dos primeras dimensiones (n, W) → queda una matriz de (n*W, F).
# - Aplica la transformación del scaler entrenado (ej. StandardScaler).
# - Restaura la forma original (n, W, F) para conservar la estructura 3D
#   necesaria en modelos secuenciales (RNN, LSTM, Transformers).
#
# Resultado: devuelve el mismo dataset 3D pero con todos los features escalados
# de manera consistente en cada ventana y paso de tiempo.
# -------------------------------------------------------------------------

def _transform_3d(X_3d, scaler):
    n, W, F = X_3d.shape
    Xf = X_3d.reshape(-1, F)
    Xs = scaler.transform(Xf).reshape(n, W, F)
#### **6.1.1. Función para elegir escalador**
    return Xs

#### **6.1.4. Función para escalar y guardar escalador**

In [47]:
def scale_and_save(
    X_train,
    X_valid=None,
    X_test=None,
    scaler_type="standard",
    scaler_path="data/windows/scaled/global_scaler.pkl",
    window_size=None,
    verbose=True
):
    """
    Escala X_train (y opcionalmente valid/test) y guarda el escalador.
    - Si X_* es 3D: (n, W, F) -> fit por feature sobre (n*W, F).
    - Si X_* es 2D: (n, W*F). Si pasás window_size=W, reescala por feature reconstruyendo 3D; si no, escala columnas tal cual.
    """
    scaler_path = Path(scaler_path)
    scaler_path.parent.mkdir(parents=True, exist_ok=True)

    scaler = _choose_scaler(scaler_type)

    if X_train.ndim == 3:
        scaler = _fit_on_3d(X_train, scaler)
        X_train_s = _transform_3d(X_train, scaler)
        X_valid_s = _transform_3d(X_valid, scaler) if X_valid is not None else None
        X_test_s  = _transform_3d(X_test,  scaler) if X_test  is not None else None

    elif X_train.ndim == 2:
        n, tot = X_train.shape
        if window_size is not None:
            assert tot % window_size == 0, "total de columnas no divisible por window_size"
            F = tot // window_size

            def to3d(X2d):
                return X2d.reshape(X2d.shape[0], window_size, F)

            Xtr3 = to3d(X_train)
            scaler = _fit_on_3d(Xtr3, scaler)

            X_train_s = _transform_3d(Xtr3, scaler).reshape(n, tot)
            X_valid_s = _transform_3d(to3d(X_valid), scaler).reshape(X_valid.shape[0], tot) if X_valid is not None else None
            X_test_s  = _transform_3d(to3d(X_test),  scaler).reshape(X_test.shape[0],  tot) if X_test  is not None else None
        else:
            scaler.fit(X_train)
            X_train_s = scaler.transform(X_train)
            X_valid_s = scaler.transform(X_valid) if X_valid is not None else None
            X_test_s  = scaler.transform(X_test)  if X_test  is not None else None
    else:
        raise ValueError("X_train debe ser 2D o 3D.")

    joblib.dump(scaler, scaler_path)

    if verbose:
        print(f"Scaler guardado en: {scaler_path}")
        print("Shapes escaladas:",
              "X_train", X_train_s.shape,
              "| X_valid", None if X_valid is None else X_valid_s.shape,
              "| X_test",  None if X_test  is None else X_test_s.shape)

    return X_train_s, X_valid_s, X_test_s, scaler

In [48]:
def scale_and_save_windows_seq2seq(
    X_train, y_train,
    X_valid, y_valid,
    X_test,  y_test,
    out_train_npz: Path,
    out_valid_npz: Path,
    out_test_npz:  Path,
    out_scaler_path: Path,
    scaler_type: str = "standard",
    verbose: bool = True,
):
    """
    Escala SOLO X (3D) con estadísticas aprendidas en train y guarda:
      - windows_train_*.npz (X escalado + y original)
      - windows_valid_*.npz
      - windows_test_*.npz
      - scaler (pkl)

    Retorna:
      X_train_s, y_train, X_valid_s, y_valid, X_test_s, y_test, scaler
    """
    out_train_npz = Path(out_train_npz); out_train_npz.parent.mkdir(parents=True, exist_ok=True)
    out_valid_npz = Path(out_valid_npz); out_valid_npz.parent.mkdir(parents=True, exist_ok=True)
    out_test_npz  = Path(out_test_npz);  out_test_npz.parent.mkdir(parents=True, exist_ok=True)
    out_scaler_path = Path(out_scaler_path); out_scaler_path.parent.mkdir(parents=True, exist_ok=True)

    # 1) Escalado (fit en train, transform valid/test)
    X_train_s, X_valid_s, X_test_s, scaler = scale_and_save(
        X_train=X_train,
        X_valid=X_valid,
        X_test=X_test,
        scaler_type=scaler_type,
        scaler_path=str(out_scaler_path),
        verbose=verbose
    )

    # 2) Guardado de ventanas escaladas (y SIN escalar)
    np.savez_compressed(out_train_npz, X=X_train_s, y=y_train)
    np.savez_compressed(out_valid_npz, X=X_valid_s, y=y_valid)
    np.savez_compressed(out_test_npz,  X=X_test_s,  y=y_test)

    if verbose:
        print(f"Windows escaladas guardadas:")
        print(f"  - train: {out_train_npz}")
        print(f"  - valid: {out_valid_npz}")
        print(f"  - test : {out_test_npz}")

    return X_train_s, y_train, X_valid_s, y_valid, X_test_s, y_test, scaler

In [52]:
import numpy as np

def check_scaled_train_windows(
    X_train: np.ndarray,
    mean_tol: float = 1e-2,
    std_tol: float = 1e-2,
) -> None:
    """
    Verifica que el escalamiento haya sido ajustado correctamente sobre TRAIN.

    Condiciones:
    - Media ≈ 0 por feature
    - Desvío estándar ≈ 1 por feature

    Parámetros
    ----------
    X_train : np.ndarray
        Ventanas de entrenamiento escaladas, shape (N, T, F)
    mean_tol : float
        Tolerancia absoluta para la media.
    std_tol : float
        Tolerancia absoluta para el desvío estándar.

    Lanza AssertionError si alguna condición no se cumple.
    """

    assert X_train.ndim == 3, "X_train debe tener forma (N, T, F)"

    # Colapsa N y T → analiza por feature
    X_flat = X_train.reshape(-1, X_train.shape[-1])

    means = X_flat.mean(axis=0)
    stds  = X_flat.std(axis=0)

    max_mean_dev = np.max(np.abs(means))
    max_std_dev  = np.max(np.abs(stds - 1.0))

    assert max_mean_dev < mean_tol, (
        f"Media fuera de tolerancia en TRAIN "
        f"(max |mean| = {max_mean_dev:.4f})"
    )

    assert max_std_dev < std_tol, (
        f"Std fuera de tolerancia en TRAIN "
        f"(max |std-1| = {max_std_dev:.4f})"
    )

    print(
        "[OK] Escalamiento TRAIN verificado | "
        f"max |mean|={max_mean_dev:.4e}, "
        f"max |std-1|={max_std_dev:.4e}"
    )

### **6.2. Escalado para horizonte de 60 minutos (`delta_pts_60`)**

In [49]:
X_train_60_z, y_train_60, X_valid_60_z, y_valid_60, X_test_60_z, y_test_60, scaler_60 = scale_and_save_windows_seq2seq(
    X_train=X_train_60, y_train=y_train_60,
    X_valid=X_valid_60, y_valid=y_valid_60,
    X_test=X_test_60,   y_test=y_test_60,
    out_train_npz=OUT_WINDOWS_60_TRAIN_Z,
    out_valid_npz=OUT_WINDOWS_60_VALID_Z,
    out_test_npz=OUT_WINDOWS_60_TEST_Z,
    out_scaler_path=OUT_SCALER_60,   # si quiere uno global para este horizonte
    scaler_type="standard",
)

Scaler guardado en: /content/drive/MyDrive/neural_profit/data/windows/scaled/scaler_60.pkl
Shapes escaladas: X_train (912, 29, 8) | X_valid (195, 29, 8) | X_test (196, 29, 8)
Windows escaladas guardadas:
  - train: /content/drive/MyDrive/neural_profit/data/windows/scaled/windows_train_60_z.npz
  - valid: /content/drive/MyDrive/neural_profit/data/windows/scaled/windows_valid_60_z.npz
  - test : /content/drive/MyDrive/neural_profit/data/windows/scaled/windows_test_60_z.npz


In [53]:
check_scaled_train_windows(X_train_60_z)

[OK] Escalamiento TRAIN verificado | max |mean|=8.2516e-16, max |std-1|=8.8818e-15


### **6.3. Escalado para horizonte de 90 minutos (`delta_pts_90`)**

In [54]:
X_train_90_z, y_train_90, X_valid_90_z, y_valid_90, X_test_90_z, y_test_90, scaler_90 = scale_and_save_windows_seq2seq(
    X_train=X_train_90, y_train=y_train_90,
    X_valid=X_valid_90, y_valid=y_valid_90,
    X_test=X_test_90,   y_test=y_test_90,
    out_train_npz=OUT_WINDOWS_90_TRAIN_Z,
    out_valid_npz=OUT_WINDOWS_90_VALID_Z,
    out_test_npz=OUT_WINDOWS_90_TEST_Z,
    out_scaler_path=OUT_SCALER_90,   # OJO: aquí lo sobrescribe si reutiliza el mismo path
    scaler_type="standard",
)

Scaler guardado en: /content/drive/MyDrive/neural_profit/data/windows/scaled/scaler_90.pkl
Shapes escaladas: X_train (912, 29, 8) | X_valid (195, 29, 8) | X_test (196, 29, 8)
Windows escaladas guardadas:
  - train: /content/drive/MyDrive/neural_profit/data/windows/scaled/windows_train_90_z.npz
  - valid: /content/drive/MyDrive/neural_profit/data/windows/scaled/windows_valid_90_z.npz
  - test : /content/drive/MyDrive/neural_profit/data/windows/scaled/windows_test_90_z.npz


In [55]:
check_scaled_train_windows(X_train_90_z)

[OK] Escalamiento TRAIN verificado | max |mean|=8.2516e-16, max |std-1|=8.8818e-15


## **7. Stage Summary Report**

### **7.1. Función para crear Summary Report**

In [ ]:
import json
import os
from pathlib import Path
from datetime import datetime

import numpy as np
import joblib


def build_stage06_summary_report(
    report_path,
    *,
    window_size: int,
    scaler_type: str,
    horizons: list,
    timezone_str: str = "America/New_York",
    time_window_str: str = "08:20–08:49",
    # mapping por horizonte: {60: {"train": Path, "valid": Path, "test": Path}, 90: {...}}
    windows_paths_by_horizon: dict,
    # mapping por horizonte: {60: Path("scaler_60.pkl"), 90: Path("scaler_90.pkl")}  o { "global": Path(...) }
    scaler_paths_by_horizon: dict,
    # mapping por horizonte: {60: {"train": X_train, "valid": X_valid, "test": X_test}, 90: {...}}
    X_by_horizon: dict | None = None,
    # mapping por horizonte: {60: {"train": y_train, "valid": y_valid, "test": y_test}, 90: {...}}
    y_by_horizon: dict | None = None,
    feature_names_by_horizon: dict | None = None,  # {60: features_60, 90: features_90}
    include_scaler_stats: bool = True,
    include_y_stats: bool = True,
    verbose: bool = True,
):
    """
    Crea el report summary del stage_06 (escalado de ventanas) y lo guarda como JSON.

    Qué incluye (resumen):
    - configuración del stage (window_size, scaler_type, horizons, timezone, franja horaria)
    - shapes y conteos por split (train/valid/test)
    - validaciones (no leakage declarado, consistencia de shapes, NaNs)
    - paths a artifacts generados (npz de ventanas escaladas, scalers)
    - stats del scaler (mean/std o min/max) si se puede cargar el scaler
    - stats de y (flatten) opcional, si se pasa y_by_horizon

    Requisitos:
    - Si no pasa X_by_horizon / y_by_horizon, el reporte igual se crea usando
      los archivos .npz de windows_paths_by_horizon para leer shapes y checks.
    """

    def _p(p):
        return str(p) if p is not None else None

    def _as_path(p):
        return p if isinstance(p, Path) else Path(str(p))

    def _safe_bool(x):
        return bool(x) if x is not None else None

    def _load_npz_shapes_and_checks(npz_path: Path):
        """
        Devuelve shapes y checks básicos leyendo el .npz:
        - X_shape, y_shape
        - has_nan_X, has_nan_y
        """
        data = np.load(npz_path)
        X = data["X"]
        y = data["y"]
        return {
            "X_shape": list(X.shape),
            "y_shape": list(y.shape),
            "has_nan_X": bool(np.isnan(X).any()),
            "has_nan_y": bool(np.isnan(y).any()),
        }

    def _flatten_stats(arr: np.ndarray):
        a = np.asarray(arr).ravel()
        return {
            "mean": float(a.mean()),
            "std": float(a.std()),
            "min": float(a.min()),
            "max": float(a.max()),
        }

    def _scaler_stats(scaler, feature_names=None):
        """
        Extrae estadísticas del scaler de forma segura.
        - StandardScaler: mean_, scale_
        - MinMaxScaler: data_min_, data_max_
        """
        out = {"scaler_class": scaler.__class__.__name__}

        if hasattr(scaler, "mean_") and hasattr(scaler, "scale_"):
            means = scaler.mean_.tolist()
            scales = scaler.scale_.tolist()
            out["type"] = "standard"
            if feature_names and len(feature_names) == len(means):
                out["per_feature"] = {
                    str(fn): {"mean": float(m), "std": float(s)}
                    for fn, m, s in zip(feature_names, means, scales)
                }
            else:
                out["mean"] = [float(x) for x in means]
                out["std"] = [float(x) for x in scales]

        elif hasattr(scaler, "data_min_") and hasattr(scaler, "data_max_"):
            mins = scaler.data_min_.tolist()
            maxs = scaler.data_max_.tolist()
            out["type"] = "minmax"
            if feature_names and len(feature_names) == len(mins):
                out["per_feature"] = {
                    str(fn): {"min": float(mi), "max": float(ma)}
                    for fn, mi, ma in zip(feature_names, mins, maxs)
                }
            else:
                out["min"] = [float(x) for x in mins]
                out["max"] = [float(x) for x in maxs]
        else:
            out["type"] = "unknown"
            out["note"] = "No se encontraron atributos estándar para extraer estadísticas."

        return out

    # -------------------------
    # Construcción del reporte
    # -------------------------
    report_path = _as_path(report_path)
    report_path.parent.mkdir(parents=True, exist_ok=True)

    report = {
        "stage": "stage_06_training_dataset_construction",
        "description": "Escalado de ventanas SEQ2SEQ diarias (X: window_size×features → y: window_size) usando estadísticas del set de entrenamiento.",
        "generated_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "config": {
            "window_size": int(window_size),
            "scaler_type": str(scaler_type),
            "scaler_scope": "train_only",
            "horizons": [int(h) for h in horizons],
            "timezone": timezone_str,
            "time_window": time_window_str,
        },
        "datasets": {},
        "checks": {
            "no_leakage_assumed": True,
            "shape_consistency": True,
            "no_nan_after_scaling": True,
        },
        "artifacts": {
            "windows_npz": {},
            "scalers": {},
        },
        "notes": [],
    }

    # Por cada horizonte, extraer info de splits
    global_shape_ok = True
    global_nan_ok = True

    for h in horizons:
        h_key = str(int(h))
        feature_names = None
        if feature_names_by_horizon is not None and h in feature_names_by_horizon:
            feature_names = list(feature_names_by_horizon[h])

        # Paths a windows escaladas
        paths = windows_paths_by_horizon.get(h, {})
        train_p = _as_path(paths.get("train"))
        valid_p = _as_path(paths.get("valid"))
        test_p  = _as_path(paths.get("test"))

        report["artifacts"]["windows_npz"][h_key] = {
            "train": _p(train_p),
            "valid": _p(valid_p),
            "test":  _p(test_p),
        }

        # Leer shapes/checks desde npz (fuente de verdad)
        info_train = _load_npz_shapes_and_checks(train_p)
        info_valid = _load_npz_shapes_and_checks(valid_p)
        info_test  = _load_npz_shapes_and_checks(test_p)

        # Validar consistencia shape: X 3D y y 2D y window_size consistente
        def _is_expected(info):
            Xs = info["X_shape"]
            ys = info["y_shape"]
            ok = True
            ok &= (len(Xs) == 3)
            ok &= (len(ys) == 2)
            if len(Xs) == 3:
                ok &= (Xs[1] == window_size)
            if len(ys) == 2:
                ok &= (ys[1] == window_size)
            if len(Xs) == 3 and len(ys) == 2:
                ok &= (Xs[0] == ys[0])
                ok &= (Xs[1] == ys[1])
            return bool(ok)

        shape_ok = _is_expected(info_train) and _is_expected(info_valid) and _is_expected(info_test)
        global_shape_ok &= shape_ok

        nan_ok = (not info_train["has_nan_X"] and not info_train["has_nan_y"] and
                  not info_valid["has_nan_X"] and not info_valid["has_nan_y"] and
                  not info_test["has_nan_X"]  and not info_test["has_nan_y"])
        global_nan_ok &= nan_ok

        # Conteos y n_features desde shapes (train como referencia)
        X_shape_train = info_train["X_shape"]
        n_features = X_shape_train[2] if len(X_shape_train) == 3 else None

        report["datasets"][h_key] = {
            "n_features": int(n_features) if n_features is not None else None,
            "splits": {
                "train": int(info_train["X_shape"][0]),
                "valid": int(info_valid["X_shape"][0]),
                "test":  int(info_test["X_shape"][0]),
            },
            "shapes": {
                "train": {"X": info_train["X_shape"], "y": info_train["y_shape"]},
                "valid": {"X": info_valid["X_shape"], "y": info_valid["y_shape"]},
                "test":  {"X": info_test["X_shape"],  "y": info_test["y_shape"]},
            },
            "checks": {
                "shape_ok": shape_ok,
                "no_nan": nan_ok,
            },
        }

        # Stats de y (opcional): si pasaron y_by_horizon, usar eso; si no, leer y desde train npz
        if include_y_stats:
            if y_by_horizon is not None and h in y_by_horizon and "train" in y_by_horizon[h]:
                ytr = y_by_horizon[h]["train"]
                yva = y_by_horizon[h].get("valid")
                yte = y_by_horizon[h].get("test")
            else:
                # cargar desde npz (solo stats, no guardar data)
                ytr = np.load(train_p)["y"]
                yva = np.load(valid_p)["y"]
                yte = np.load(test_p)["y"]

            report["datasets"][h_key]["y_stats"] = {
                "train": _flatten_stats(ytr),
                "valid": _flatten_stats(yva),
                "test":  _flatten_stats(yte),
            }

        # Stats del scaler (opcional): cargar pkl si existe
        scaler_p = scaler_paths_by_horizon.get(h) or scaler_paths_by_horizon.get(h_key) or scaler_paths_by_horizon.get("global")
        if scaler_p is not None:
            scaler_p = _as_path(scaler_p)
            report["artifacts"]["scalers"][h_key] = _p(scaler_p)

            if include_scaler_stats and scaler_p.exists():
                try:
                    scaler_obj = joblib.load(scaler_p)
                    report["datasets"][h_key]["scaler_stats"] = _scaler_stats(scaler_obj, feature_names=feature_names)
                except Exception as e:
                    report["datasets"][h_key]["scaler_stats_error"] = str(e)
        else:
            report["artifacts"]["scalers"][h_key] = None
            report["notes"].append(f"No se proporcionó scaler_path para horizonte {h_key}.")

        # Incluir features usadas (opcional, útil para trazabilidad)
        if feature_names is not None:
            report["datasets"][h_key]["feature_names"] = feature_names

    # Checks globales
    report["checks"]["shape_consistency"] = bool(global_shape_ok)
    report["checks"]["no_nan_after_scaling"] = bool(global_nan_ok)

    # Nota sobre y no escalada (decisión de diseño)
    report["notes"].append("El target (y) se guarda sin escalar para mantener unidades en puntos (delta_pts).")
    report["notes"].append("El scaler se ajusta únicamente con X_train (aplanando n_days×window_size) para evitar leakage.")

    # Guardar
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    if verbose:
        print(f"Report stage_06 guardado en: {report_path}")

    return report


### **7.2. Función imprimiar Summary Report**

In [ ]:
from typing import Dict, Any


def print_stage06_summary_pretty(summary: Dict[str, Any]) -> None:
    """
    Pretty print del summary dict de stage_06 (window scaling).
    """
    if not summary:
        print("Summary vacío (stage_06).")
        return

    config = summary.get("config", {})
    datasets = summary.get("datasets", {})
    checks = summary.get("checks", {})
    artifacts = summary.get("artifacts", {})

    print("\n" + "=" * 78)
    print("STAGE_06 – TRAINING DATASET CONSTRUCTION (SEQ2SEQ MNQ)")
    print("=" * 78)

    # ------------------------------------------------------------------
    # Configuración general
    # ------------------------------------------------------------------
    print("\nConfiguración")
    print("-" * 78)
    print(f"Window size        : {config.get('window_size')}")
    print(f"Scaler type        : {config.get('scaler_type')}")
    print(f"Scaler scope       : {config.get('scaler_scope')}")
    print(f"Horizontes         : {config.get('horizons')}")
    print(f"Timezone           : {config.get('timezone')}")
    print(f"Ventana horaria    : {config.get('time_window')}")

    # ------------------------------------------------------------------
    # Datasets por horizonte
    # ------------------------------------------------------------------
    for h, info in datasets.items():
        print("\n" + "-" * 78)
        print(f"Horizonte {h} minutos")
        print("-" * 78)

        print(f"Features           : {info.get('n_features')}")

        splits = info.get("splits", {})
        print("Splits (n_days)")
        print(f"  Train            : {splits.get('train')}")
        print(f"  Valid            : {splits.get('valid')}")
        print(f"  Test             : {splits.get('test')}")

        shapes = info.get("shapes", {})
        if shapes:
            print("\nShapes")
            for split_name, sh in shapes.items():
                print(f"  {split_name:<6} -> X: {sh.get('X')} | y: {sh.get('y')}")

        # Checks por horizonte
        h_checks = info.get("checks", {})
        if h_checks:
            print("\nChecks")
            print(f"  Shape OK         : {h_checks.get('shape_ok')}")
            print(f"  No NaN           : {h_checks.get('no_nan')}")

        # Estadísticas del target
        y_stats = info.get("y_stats")
        if y_stats:
            print("\nTarget (y) stats – flatten")
            for split_name, st in y_stats.items():
                print(
                    f"  {split_name:<6} -> "
                    f"mean={st.get('mean'):.6f} | "
                    f"std={st.get('std'):.6f} | "
                    f"min={st.get('min'):.6f} | "
                    f"max={st.get('max'):.6f}"
                )

        # Estadísticas del scaler
        scaler_stats = info.get("scaler_stats")
        if scaler_stats:
            print("\nScaler stats")
            print(f"  Class            : {scaler_stats.get('scaler_class')}")
            print(f"  Type             : {scaler_stats.get('type')}")

            per_feat = scaler_stats.get("per_feature")
            if per_feat:
                print("  Por feature:")
                for fname, vals in per_feat.items():
                    if "mean" in vals:
                        print(
                            f"    {fname:<20} "
                            f"mean={vals['mean']:.6f} | std={vals['std']:.6f}"
                        )
                    else:
                        print(
                            f"    {fname:<20} "
                            f"min={vals['min']:.6f} | max={vals['max']:.6f}"
                        )

    # ------------------------------------------------------------------
    # Checks globales
    # ------------------------------------------------------------------
    print("\n" + "-" * 78)
    print("Checks globales")
    print("-" * 78)
    print(f"No leakage asumido : {checks.get('no_leakage_assumed')}")
    print(f"Shape consistente  : {checks.get('shape_consistency')}")
    print(f"No NaN post-scale  : {checks.get('no_nan_after_scaling')}")

    # ------------------------------------------------------------------
    # Artifacts
    # ------------------------------------------------------------------
    print("\n" + "-" * 78)
    print("Artifacts generados")
    print("-" * 78)

    win_art = artifacts.get("windows_npz", {})
    for h, paths in win_art.items():
        print(f"Horizonte {h}:")
        print(f"  Train  : {paths.get('train')}")
        print(f"  Valid  : {paths.get('valid')}")
        print(f"  Test   : {paths.get('test')}")

    scalers = artifacts.get("scalers", {})
    if scalers:
        print("\nScalers")
        for h, p in scalers.items():
            print(f"  {h:<6} : {p}")

    # ------------------------------------------------------------------
    # Notas
    # ------------------------------------------------------------------
    notes = summary.get("notes", [])
    if notes:
        print("\n" + "-" * 78)
        print("Notas")
        print("-" * 78)
        for n in notes:
            print(f"- {n}")

    print("\n" + "=" * 78)


### **7.3. Creación de Summary Report**

In [ ]:
windows_paths_by_horizon = {
    60: {"train": OUT_WINDOWS_60_TRAIN_Z, "valid": OUT_WINDOWS_60_VALID_Z, "test": OUT_WINDOWS_60_TEST_Z},
    90: {"train": OUT_WINDOWS_90_TRAIN_Z, "valid": OUT_WINDOWS_90_VALID_Z, "test": OUT_WINDOWS_90_TEST_Z},
}

# Recomendación: un scaler por horizonte (para no sobrescribir). Si usted usa uno global, use {"global": OUT_SCALER}
scaler_paths_by_horizon = {
    60: OUT_SCALER_60,
    90: OUT_SCALER_90,
}

report = build_stage06_summary_report(
    report_path=OUT_SUMMARY,
    window_size=30,
    scaler_type="standard",
    horizons=[60, 90],
    timezone_str="America/New_York",
    time_window_str="08:20–08:49",
    windows_paths_by_horizon=windows_paths_by_horizon,
    scaler_paths_by_horizon=scaler_paths_by_horizon,
    feature_names_by_horizon={60: features_60, 90: features_90},  # opcional
    include_scaler_stats=True,
    include_y_stats=True,
    verbose=True,
)


In [ ]:
print_stage06_summary_pretty(report)

## **8. Alineamiento con libro de ML**

## Preprocesamiento y escalamiento de datos

Este paso se realiza **después del split temporal** y **antes del entrenamiento del modelo**, de acuerdo con las buenas prácticas de *Machine Learning* para series temporales financieras.

---

### Aspectos correctamente alineados

1. **Orden del proceso**
   - El split temporal se realiza **antes** del escalamiento.
   - El escalamiento se realiza **antes** del entrenamiento del modelo.

2. **Regla crítica anti-leakage**
   - El *scaler* se ajusta **exclusivamente con el conjunto de entrenamiento (TRAIN)**.
   - Los conjuntos de *validation* y *test* se transforman utilizando ese mismo *scaler*,
     sin volver a ajustarlo.

3. **Escalamiento aplicado únicamente a features**
   - Las variables de entrada (OHLCV / features) son escaladas.
   - El target (`delta_pts_H`) **no se escala**, preservando su interpretación económica en unidades de puntos.

4. **Consistencia entre horizontes**
   - Se utilizan *scalers* independientes para **H = 60** y **H = 90**.
   - No se mezclan estadísticas entre distintos horizontes de predicción.

5. **Persistencia**
   - Los *scalers* se guardan para asegurar reproducibilidad.
   - Son reutilizables en etapas posteriores de inferencia.

---

### Criterio de escalamiento

Se utiliza **StandardScaler (z-score)**, ajustado exclusivamente con el conjunto de
entrenamiento, por su adecuación a modelos sensibles a la escala.

El escalamiento se realiza **por feature de forma global**, y no de manera independiente
por jornada bursátil.

---

### Verificación automática del escalamiento

Se realizó un *sanity check* sobre el conjunto de entrenamiento escalado, verificando que:

- la media por feature sea aproximadamente 0,
- el desvío estándar por feature sea aproximadamente 1,

únicamente en el conjunto **TRAIN**.

Resultado para **H = 60**:

